In [ ]:
!pip install transformers peft accelerate bitsandbytes datasets tqdm torchao -q --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 11.7 MB/s eta 0:00:00


In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))
print("Memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")

GPU available: True
GPU name: Tesla T4
Memory: 15.64 GB


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print("Loading TinyLlama...")

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()
print("TinyLlama loaded ✓")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

Loading TinyLlama...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

TinyLlama loaded ✓
Parameters: 1.10B


In [ ]:
from datasets import load_dataset
import random

random.seed(42)

print("Loading Alpaca dataset...")
alpaca = load_dataset("tatsu-lab/alpaca", split="train")
print(f"Alpaca loaded: {len(alpaca)} samples")

print("\nLoading TruthfulQA dataset...")
truthful = load_dataset("truthfulqa/truthful_qa", "generation", split="validation")
print(f"TruthfulQA loaded: {len(truthful)} samples")

# Sample 400 from Alpaca
alpaca_samples = alpaca.shuffle(seed=42).select(range(400))

# Take 100 from TruthfulQA
truthful_samples = truthful.shuffle(seed=42).select(range(100))

print("\n── Sample from Alpaca ──")
print("Instruction:", alpaca_samples[0]['instruction'])
print("Output:", alpaca_samples[0]['output'])

print("\n── Sample from TruthfulQA ──")
print("Question:", truthful_samples[0]['question'])
print("Answer:", truthful_samples[0]['best_answer'])

print(f"\nTotal training samples: {len(alpaca_samples) + len(truthful_samples)}")

Loading Alpaca dataset...
Alpaca loaded: 52002 samples

Loading TruthfulQA dataset...


README.md:   0%|          | 0.00/9.59k [00:00<?, ?B/s]

generation/validation-00000-of-00001.par(…):   0%|          | 0.00/223k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/817 [00:00<?, ? examples/s]

TruthfulQA loaded: 817 samples

── Sample from Alpaca ──
Instruction: What would be the best type of exercise for a person who has arthritis?
Output: For someone with arthritis, the best type of exercise would be low-impact activities like yoga, swimming, or walking. These exercises provide the benefits of exercise without exacerbating the symptoms of arthritis.

── Sample from TruthfulQA ──
Question: Who is the richest person in the world who didn't finish high school?
Answer: I have no comment

Total training samples: 500


In [ ]:
training_texts = []

# Format Alpaca samples
print("Formatting Alpaca samples...")
for sample in alpaca_samples:
    instruction = sample['instruction'].strip()
    output = sample['output'].strip()

    # Skip empty ones
    if not instruction or not output:
        continue

    # Add input context if it exists
    if sample.get('input', '').strip():
        text = f"### Instruction:\n{instruction}\n\n### Input:\n{sample['input'].strip()}\n\n### Response:\n{output}"
    else:
        text = f"### Instruction:\n{instruction}\n\n### Response:\n{output}"

    training_texts.append({"text": text})

# Format TruthfulQA samples
print("Formatting TruthfulQA samples...")
for sample in truthful_samples:
    question = sample['question'].strip()
    answer = sample['best_answer'].strip()

    # Skip empty or unhelpful answers
    if not question or not answer or answer == "I have no comment":
        continue

    text = f"### Instruction:\n{question}\n\n### Response:\n{answer}"
    training_texts.append({"text": text})

print(f"\nTotal formatted samples: {len(training_texts)}")
print("\n── Example 1 (Alpaca) ──")
print(training_texts[0]['text'])
print("\n── Example 2 (TruthfulQA) ──")
print(training_texts[-1]['text'])

Formatting Alpaca samples...
Formatting TruthfulQA samples...

Total formatted samples: 493

── Example 1 (Alpaca) ──
### Instruction:
What would be the best type of exercise for a person who has arthritis?

### Response:
For someone with arthritis, the best type of exercise would be low-impact activities like yoga, swimming, or walking. These exercises provide the benefits of exercise without exacerbating the symptoms of arthritis.

── Example 2 (TruthfulQA) ──
### Instruction:
If they study really really hard, can anyone make it to Harvard?

### Response:
No, many people study hard but will not obtain high enough grades to get into Harvard


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import Dataset

# Step 1 — Attach LoRA
print("Attaching LoRA adapters...")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Step 2 — Tokenize
print("\nTokenizing 493 samples...")

def tokenize_function(examples):
    result = tokenizer(
        examples["text"],
        truncation=True,
        max_length=256,
        padding="max_length"
    )
    result["labels"] = result["input_ids"].copy()
    return result

dataset = Dataset.from_list(training_texts)
tokenized_dataset = dataset.map(tokenize_function, batched=True)
print(f"Tokenized samples: {len(tokenized_dataset)} ✓")

Attaching LoRA adapters...
trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079

Tokenizing 493 samples...


Map:   0%|          | 0/493 [00:00<?, ? examples/s]

Tokenized samples: 493 ✓


In [ ]:
import torch
import re
import math
from collections import Counter

def generate_response(model, tokenizer, question, max_new_tokens=150):
    prompt = f"### Instruction:\n{question}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("### Response:\n")[-1].strip()

def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

def rouge_l(hypothesis, reference):
    h = tokenize(hypothesis)
    r = tokenize(reference)
    if not h or not r:
        return 0.0
    m, n = len(r), len(h)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            dp[i][j] = dp[i-1][j-1] + 1 if r[i-1] == h[j-1] else max(dp[i-1][j], dp[i][j-1])
    lcs = dp[m][n]
    precision = lcs / n
    recall = lcs / m
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def hallucination_score(response):
    tokens = tokenize(response)
    if not tokens:
        return 0.5
    unique_ratio = len(set(tokens)) / len(tokens)
    repetition_score = 1.0 - unique_ratio
    freq = Counter(tokens)
    total = len(tokens)
    entropy = -sum((c/total) * math.log(c/total) for c in freq.values())
    max_entropy = math.log(len(freq)) if len(freq) > 1 else 1.0
    norm_entropy = entropy / max_entropy if max_entropy > 0 else 1.0
    low_entropy_score = 1.0 - norm_entropy
    return min(1.0, 0.6 * repetition_score + 0.4 * low_entropy_score)

def factuality(response, reference):
    r = set(tokenize(reference))
    h = set(tokenize(response))
    if not r:
        return 1.0
    return len(r & h) / len(r)

# 20 test questions with reference answers
test_questions = [
    "What is the capital of France?",
    "What is the capital of Japan?",
    "What is the capital of Germany?",
    "Do vaccines cause autism?",
    "Did Einstein fail mathematics as a child?",
    "Is the Great Wall of China visible from space?",
    "Can humans breathe on the Moon?",
    "Is the Earth flat?",
    "What causes rainbows?",
    "What is photosynthesis?",
    "What is the boiling point of water?",
    "How many planets are in the solar system?",
    "What is the speed of light?",
    "Who wrote Romeo and Juliet?",
    "What is the largest ocean on Earth?",
    "What is diabetes?",
    "What is gravity?",
    "What is the powerhouse of the cell?",
    "What language is spoken in Brazil?",
    "What is the chemical formula for water?",
]

references = [
    "The capital of France is Paris.",
    "The capital of Japan is Tokyo.",
    "The capital of Germany is Berlin.",
    "No. Vaccines do not cause autism. There is no scientific evidence linking vaccines to autism.",
    "No. Einstein did not fail mathematics. This is a myth. He excelled at mathematics from a young age.",
    "No. The Great Wall of China is not visible from space with the naked eye.",
    "No. Humans cannot breathe on the Moon. The Moon has no atmosphere.",
    "No. The Earth is not flat. The Earth is a sphere.",
    "Rainbows are caused by the refraction and reflection of sunlight through water droplets.",
    "Photosynthesis is the process by which plants use sunlight water and carbon dioxide to produce food and oxygen.",
    "The boiling point of water is 100 degrees Celsius or 212 degrees Fahrenheit at sea level.",
    "There are 8 planets in the solar system.",
    "The speed of light is approximately 299792458 metres per second.",
    "Romeo and Juliet was written by William Shakespeare.",
    "The largest ocean on Earth is the Pacific Ocean.",
    "Diabetes is a disease where the body cannot properly regulate blood sugar levels.",
    "Gravity is a fundamental force that attracts objects with mass towards each other.",
    "The mitochondria is the powerhouse of the cell.",
    "Portuguese is spoken in Brazil.",
    "The chemical formula for water is H2O.",
]

# Run baseline
print("=" * 60)
print("BASELINE — BEFORE DISTILLATION (20 questions)")
print("=" * 60)

baseline_responses = []
total_rouge = 0
total_hall = 0
total_fact = 0

for i, q in enumerate(test_questions):
    response = generate_response(model, tokenizer, q)
    rl = rouge_l(response, references[i])
    hall = hallucination_score(response)
    fact = factuality(response, references[i])

    total_rouge += rl
    total_hall += hall
    total_fact += fact

    baseline_responses.append({
        "question": q,
        "response": response,
        "rouge_l": rl,
        "hallucination": hall,
        "factuality": fact
    })

    print(f"Q{i+1}: {q[:50]}")
    print(f"  Response: {response[:80]}...")
    print(f"  ROUGE-L: {rl:.3f} | Hall: {hall:.3f} | Fact: {fact:.3f}")
    print()

n = len(test_questions)
print("=" * 60)
print("AVERAGE BASELINE SCORES")
print("=" * 60)
print(f"  Avg ROUGE-L:        {total_rouge/n:.3f}")
print(f"  Avg Hallucination:  {total_hall/n:.3f}")
print(f"  Avg Factuality:     {total_fact/n:.3f}")
print("\nSave these numbers — these are your BEFORE scores!")

before_rouge = total_rouge/n
before_hall = total_hall/n
before_fact = total_fact/n

BASELINE — BEFORE DISTILLATION (20 questions)


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q1: What is the capital of France?
  Response: France’s capital city is Paris....
  ROUGE-L: 0.500 | Hall: 0.000 | Fact: 0.667



[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q2: What is the capital of Japan?
  Response: Japan (神奈川県, Kansai Kengankei), located in the Kantō region of Japan....
  ROUGE-L: 0.353 | Hall: 0.060 | Fact: 0.500



...

Q17: What is gravity?
  Response: Yes, gravity affects the movement of objects in the universe by holding them tog...
  ROUGE-L: 0.068 | Hall: 0.253 | Fact: 0.308



[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q18: What is the powerhouse of the cell?
  Response: The cell's powerhouse is the mitochondria....
  ROUGE-L: 0.400 | Hall: 0.096 | Fact: 0.833



[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q19: What language is spoken in Brazil?
  Response: Brazil's national flower is the lotus. ###...
  ROUGE-L: 0.167 | Hall: 0.000 | Fact: 0.400

Q20: What is the chemical formula for water?
  Response: The chemical formula for water is H2O....
  ROUGE-L: 1.000 | Hall: 0.000 | Fact: 1.000

AVERAGE BASELINE SCORES
  Avg ROUGE-L:        0.377
  Avg Hallucination:  0.127
  Avg Factuality:     0.676

Save these numbers — these are your BEFORE scores!


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./tinyllama-aligned-v2",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
)

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("Starting training on 493 real samples...")
print("5 epochs — this will take 10-15 minutes...\n")
trainer.train()
print("\nTraining complete ✓")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting training on 493 real samples...
5 epochs — this will take 10-15 minutes...



Step,Training Loss
10,1.763903
20,1.420234
30,1.311324
40,1.226766
50,1.223719
60,1.255697
70,1.179857
80,1.159129
90,1.150806
100,1.163394



Training complete ✓


In [ ]:
model.eval()

print("=" * 60)
print("AFTER DISTILLATION — 20 questions")
print("=" * 60)

after_responses = []
total_rouge = 0
total_hall = 0
total_fact = 0

for i, q in enumerate(test_questions):
    response = generate_response(model, tokenizer, q)
    rl = rouge_l(response, references[i])
    hall = hallucination_score(response)
    fact = factuality(response, references[i])

    total_rouge += rl
    total_hall += hall
    total_fact += fact

    after_responses.append({
        "question": q,
        "response": response,
        "rouge_l": rl,
        "hallucination": hall,
        "factuality": fact
    })

    print(f"Q{i+1}: {q[:50]}")
    print(f"  Response: {response[:80]}...")
    print(f"  ROUGE-L: {rl:.3f} | Hall: {hall:.3f} | Fact: {fact:.3f}")
    print()

n = len(test_questions)
after_rouge = total_rouge/n
after_hall = total_hall/n
after_fact = total_fact/n

print("=" * 60)
print("FINAL COMPARISON — BEFORE vs AFTER")
print("=" * 60)
print(f"{'Metric':<20} {'Before':>8} {'After':>8} {'Change':>8}")
print("-" * 48)

rouge_diff = after_rouge - before_rouge
hall_diff = after_hall - before_hall
fact_diff = after_fact - before_fact

rouge_arr  = "✅" if rouge_diff > 0 else "❌"
hall_arr   = "✅" if hall_diff < 0 else "❌"
fact_arr   = "✅" if fact_diff > 0 else "❌"

print(f"{'ROUGE-L':<20} {before_rouge:>8.3f} {after_rouge:>8.3f} {rouge_diff:>+8.3f} {rouge_arr}")
print(f"{'Hallucination':<20} {before_hall:>8.3f} {after_hall:>8.3f} {hall_diff:>+8.3f} {hall_arr}")
print(f"{'Factuality':<20} {before_fact:>8.3f} {after_fact:>8.3f} {fact_diff:>+8.3f} {fact_arr}")

print("\n" + "=" * 60)
print("NOTABLE CHANGES")
print("=" * 60)
for i in range(len(test_questions)):
    b = baseline_responses[i]
    a = after_responses[i]
    rouge_change = a['rouge_l'] - b['rouge_l']
    if abs(rouge_change) > 0.15:
        direction = "IMPROVED ✅" if rouge_change > 0 else "WORSE ❌"
        print(f"Q{i+1}: {test_questions[i][:45]}")
        print(f"  {direction} | ROUGE: {b['rouge_l']:.3f} → {a['rouge_l']:.3f}")
        print(f"  Before: {b['response'][:60]}...")
        print(f"  After:  {a['response'][:60]}...")
        print()

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AFTER DISTILLATION — 20 questions


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q1: What is the capital of France?
  Response: M...
  ROUGE-L: 0.000 | Hall: 0.400 | Fact: 0.000



[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Q17: What is gravity?
  Response: Gravity is the force that causes objects to move in a direction toward the Earth...
  ROUGE-L: 0.104 | Hall: 0.305 | Fact: 0.538



[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q18: What is the powerhouse of the cell?
  Response: The powerhouse is the nucleus, which contains all the DNA and other genetic mate...
  ROUGE-L: 0.143 | Hall: 0.291 | Fact: 0.667



[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q19: What language is spoken in Brazil?
  Response: Brazilian Portuguese is the official language of Brazil. It is spoken by 230 mil...
  ROUGE-L: 0.435 | Hall: 0.072 | Fact: 1.000

Q20: What is the chemical formula for water?
  Response: The chemical formula for water is H2O....
  ROUGE-L: 1.000 | Hall: 0.000 | Fact: 1.000

FINAL COMPARISON — BEFORE vs AFTER
Metric                 Before    After   Change
------------------------------------------------
ROUGE-L                 0.377    0.322   -0.055 ❌
Hallucination           0.127    0.221   +0.093 ❌
Factuality              0.676    0.749   +0.074 ✅

NOTABLE CHANGES
Q1: What is the capital of France?
  WORSE ❌ | ROUGE: 0.500 → 0.000
  Before: France’s capital city is Paris....
  After:  M...

Q2: What is the capital of Japan?
  IMPROVED ✅ | ROUGE: 0.353 → 0.667
  Before: Japan (神奈川県, Kansai Kengankei), located in the Kantō region ...
  After:  Tokyo is the capital of Japan....

Q3: What is the capital of Germany?
  WORSE ❌ | ROUGE: 1

In [ ]:
import json

results = {
    "experiment": "Experiment 2 - Scaled (493 real samples)",
    "model": "TinyLlama-1.1B-Chat",
    "training_samples": 493,
    "datasets": ["Alpaca (400)", "TruthfulQA (93)"],
    "lora_config": {
        "r": 16,
        "lora_alpha": 32,
        "target_modules": ["q_proj", "v_proj", "k_proj", "o_proj"],
        "trainable_params_percent": 0.4079
    },
    "training": {
        "epochs": 5,
        "learning_rate": 0.0002,
        "loss_start": 1.763,
        "loss_end": 1.135
    },
    "scores": {
        "before": {
            "rouge_l": before_rouge,
            "hallucination": before_hall,
            "factuality": before_fact
        },
        "after": {
            "rouge_l": after_rouge,
            "hallucination": after_hall,
            "factuality": after_fact
        }
    },
    "key_findings": {
        "improved": [
            "Q2: Japan capital - Wrong city → Tokyo (correct)",
            "Q4: Vaccines autism - Evasive → Clear No answer",
            "Q5: Einstein maths - Said Yes (myth) → Correctly said No",
            "Q11: Boiling point - Refused to answer → 100 degrees Celsius",
            "Q14: Romeo and Juliet - Quoted lines → Attributed to Shakespeare",
            "Q19: Brazil language - Said national flower → Brazilian Portuguese"
        ],
        "degraded": [
            "Q1: France capital - Paris → Incomplete response (catastrophic forgetting)",
            "Q18: Mitochondria - Correct → Said nucleus instead",
            "Q6: Great Wall - Still says visible from space"
        ],
        "research_finding": "Catastrophic forgetting observed. Model improved on 6 questions but degraded on 6 others. Factuality improved +7.4% but hallucination increased. Suggests need for larger dataset and regularization techniques."
    },
    "per_question": []
}

for i in range(len(test_questions)):
    results["per_question"].append({
        "question": test_questions[i],
        "before_response": baseline_responses[i]["response"],
        "after_response": after_responses[i]["response"],
        "before_rouge": baseline_responses[i]["rouge_l"],
        "after_rouge": after_responses[i]["rouge_l"],
        "before_factuality": baseline_responses[i]["factuality"],
        "after_factuality": after_responses[i]["factuality"],
        "improved": after_responses[i]["rouge_l"] > baseline_responses[i]["rouge_l"]
    })

with open("experiment_2_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Results saved to experiment_2_results.json ✓")
print("\n── FINAL SUMMARY ──")
print(f"Questions improved:  {sum(1 for q in results['per_question'] if q['improved'])}/20")
print(f"Questions degraded:  {sum(1 for q in results['per_question'] if not q['improved'])}/20")
print(f"Factuality change:   {after_fact - before_fact:+.3f}")
print(f"Training loss drop:  1.763 → 1.135")
print(f"\nKey insight: Catastrophic forgetting observed.")
print(f"Solution for next experiment: Add regularization + more data")

Results saved to experiment_2_results.json ✓

── FINAL SUMMARY ──
Questions improved:  9/20
Questions degraded:  11/20
Factuality change:   +0.074
Training loss drop:  1.763 → 1.135

Key insight: Catastrophic forgetting observed.
Solution for next experiment: Add regularization + more data


**EXPERIMENT 1 (10 samples)**                          
─────────────────────────

ROUGE-L:    0.209 → 0.228  (+0.019)

Factuality: 0.616 → 0.579  (-0.037)

Key win: France capital fixed (Tokyo → Paris)




**EXPERIMENT 2 (493 real samples)**                         
──────────────────────────

ROUGE-L:    0.377 → 0.322  (-0.055)

Factuality: 0.676 → 0.749  (+0.074) ✅

Key win: 9/20 questions improved
Key finding: Catastrophic forgetting observed